# Neuromod dose-response pilot (GPU)

Runs the four hero threads on **SDXL-Turbo** (open weights, no token) and prints the
**decision matrix** that ranks the most-stunning thread for the talk, plus the vitals-monitor
hero clip.

**First:** `Runtime -> Change runtime type -> Hardware accelerator: GPU` (a free **T4** is enough).
Total run is a few minutes plus one-time model downloads (~a few GB).

In [ ]:
!nvidia-smi -L   # confirm a GPU is attached

## 1. Get the code + deps
SDXL-Turbo/diffusers + the lazy-optional metric backends (open_clip, lpips). Colab already
ships torch/numpy/pandas/scipy/matplotlib/scikit-learn/Pillow.

In [ ]:
!git clone -q https://github.com/cneckar/neuromod-llm-poc.git
%cd neuromod-llm-poc
# All the pilot code is on main now.
!git checkout -q main
!pip install -q 'diffusers>=0.27' transformers accelerate open_clip_torch lpips scikit-image imageio seaborn

## 2. Run the pilot (N=16 seeds, coarse dose grid)
Sweeps each thread's packs 0.0->1.0, computes the metric bundle per generation, and scores the
four threads on **monotonic dose-response strength** (max |Spearman ρ|, FDR-gated) plus visual
drama, security relevance and novelty. Thread A (Latent Specter) uses CLIP concept proximity;
thread B (safety) now runs the independent **SafetyOracle** on every benign generation (real
`safety_flag_oracle` / proximity metrics — no longer NaN); thread C (Cocaine Crunch) and the
vitals proxy run fully. Images are saved and the winning thread's **hero montage** is built
automatically. (~a few min + one-time model download.)

In [ ]:
!python scripts/run_pilot.py --model sdxl-turbo --seeds 16 \
    --intensities 0.0,0.25,0.5,0.75,1.0 --outdir outputs/pilot

## 3. The decision matrix — which thread is the headline?

In [ ]:
import pandas as pd
pd.read_csv('outputs/pilot/decision_matrix.csv')

## 3b. The winning thread's hero figure
The pilot builds the headline thread's money-shot montage (ghost montage / collapse grid) from
the saved top-dose images and writes it to `outputs/pilot/hero_<thread>.png`.

In [ ]:
import glob
from IPython.display import Image, display
heroes = sorted(glob.glob('outputs/pilot/hero_*.png'))
if heroes:
    for h in heroes:
        print(h); display(Image(h))
else:
    print('No hero montage (headline thread has no montage builder, e.g. vitals_monitor).')

## 4. Dose-response curves (ribbon plots + EC50/monotonicity) for a thread
Example: the Cocaine Crunch (mode-collapse) thread. Swap the CSV for any other thread.

In [ ]:
!python analysis/dose_response_stats.py --in outputs/pilot/mode_collapse.csv \
    --outdir outputs/pilot/mc_stats --plots
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('outputs/pilot/mc_stats/plots/*.png'))[:4]:
    display(Image(p))

## 5. The vitals-monitor hero clip (single seed, LSD 0->1)

In [ ]:
!python demo/vitals_monitor.py --pack lsd --prompt 'a tree' --seed 42 \
    --steps 0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0 --outdir outputs/vitals
from IPython.display import Image
Image('outputs/vitals/vitals_monitor.gif')

## 6. Scale up + download
For the full study (#12): raise `--seeds 100` and use the fine grid
`--intensities 0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0`. The runner is resumable, so you can
stop/restart. Then pull the artifacts down:

In [ ]:
!zip -qr pilot_outputs.zip outputs/pilot outputs/vitals
from google.colab import files
files.download('pilot_outputs.zip')